# 1. Input

The first stage of the solution is to load the graphs into Python.

Both levels provide the graph as an **adjacency list**. Each dictionary key represents a node, while the associated list contains the nodes that can be reached directly from it.

Level 1 uses:

```text
node + weight
```

where `weight` represents travel time.

Level 2 uses:

```text
node + time + risk
```

where the effective edge cost is:

$$
\text{effective cost} = \text{time} + \text{risk}
$$

The graphs are undirected, so each connection is represented in both directions.

We will store the input exactly as provided by the challenge and define the starting point, destination, and required stations separately.

## Level 1 Input

The Level 1 objective is to travel from `A` to `B` using the minimum total travel time.

```python
level1_graph = {
    "A": [
        {"node": "C", "weight": 4},
        {"node": "D", "weight": 2}
    ],
    "B": [
        {"node": "E", "weight": 4},
        {"node": "F", "weight": 7}
    ],
    "C": [
        {"node": "A", "weight": 4},
        {"node": "D", "weight": 1},
        {"node": "E", "weight": 5}
    ],
    "D": [
        {"node": "A", "weight": 2},
        {"node": "C", "weight": 1},
        {"node": "E", "weight": 3},
        {"node": "F", "weight": 6}
    ],
    "E": [
        {"node": "C", "weight": 5},
        {"node": "D", "weight": 3},
        {"node": "F", "weight": 2},
        {"node": "B", "weight": 4}
    ],
    "F": [
        {"node": "D", "weight": 6},
        {"node": "E", "weight": 2},
        {"node": "B", "weight": 7}
    ]
}

level1_start = "A"
level1_end = "B"
```


### 1. Input for level 1

For Level 1, the graph is provided in the input file `1.txt`.

The input describes a **weighted, undirected graph** using an adjacency list. Each node has a list of neighbouring nodes, and each connection has a `weight` representing the travel time between the two nodes.

The requirements for this level are:

* **Start node:** `A`
* **End node:** `B`
* **Number of nodes:** `6`
* **Required stops:** None
* **Time weighting:** Yes
* **Risk weighting:** No

Our first task is therefore to load the contents of `1.txt` into Python.

Because the input is provided as JSON, we can use Python's built-in `json` library to convert the contents of the file into a Python dictionary.

We will not perform any shortest-path calculations yet. At this stage, we are only concerned with **loading and inspecting the input data**.

The graph will later be passed to Dijkstra's algorithm during the processing stage.


In [14]:
import json

with open("1.txt", "r") as file:
    level1_input = json.load(file)

level1_input

{'level': 1,
 'nodes': 6,
 'edges': 9,
 'start': 'A',
 'end': 'B',
 'adjacency_list': {'A': [{'node': 'C', 'weight': 4},
   {'node': 'D', 'weight': 2}],
  'B': [{'node': 'E', 'weight': 4}, {'node': 'F', 'weight': 7}],
  'C': [{'node': 'A', 'weight': 4},
   {'node': 'D', 'weight': 1},
   {'node': 'E', 'weight': 5}],
  'D': [{'node': 'A', 'weight': 2},
   {'node': 'C', 'weight': 1},
   {'node': 'E', 'weight': 3},
   {'node': 'F', 'weight': 6}],
  'E': [{'node': 'C', 'weight': 5},
   {'node': 'D', 'weight': 3},
   {'node': 'F', 'weight': 2},
   {'node': 'B', 'weight': 4}],
  'F': [{'node': 'D', 'weight': 6},
   {'node': 'E', 'weight': 2},
   {'node': 'B', 'weight': 7}]}}

In [15]:
level1 = level1_input["level"]
num_nodes = level1_input["nodes"]
num_edges = level1_input["edges"]

start_node = level1_input["start"]
end_node = level1_input["end"]

level1_graph = level1_input["adjacency_list"]

In [17]:
print("Level:", level1)
print("Number of nodes:", num_nodes)
print("Number of edges:", num_edges)
print("Start node:", start_node)
print("End node:", end_node)
level1_graph

Level: 1
Number of nodes: 6
Number of edges: 9
Start node: A
End node: B


{'A': [{'node': 'C', 'weight': 4}, {'node': 'D', 'weight': 2}],
 'B': [{'node': 'E', 'weight': 4}, {'node': 'F', 'weight': 7}],
 'C': [{'node': 'A', 'weight': 4},
  {'node': 'D', 'weight': 1},
  {'node': 'E', 'weight': 5}],
 'D': [{'node': 'A', 'weight': 2},
  {'node': 'C', 'weight': 1},
  {'node': 'E', 'weight': 3},
  {'node': 'F', 'weight': 6}],
 'E': [{'node': 'C', 'weight': 5},
  {'node': 'D', 'weight': 3},
  {'node': 'F', 'weight': 2},
  {'node': 'B', 'weight': 4}],
 'F': [{'node': 'D', 'weight': 6},
  {'node': 'E', 'weight': 2},
  {'node': 'B', 'weight': 7}]}

# 2. Processing Data

Now that the Level 1 graph has been loaded, we can process the graph to find the shortest route from the start node `A` to the destination node `B`.

The appropriate algorithm for this problem is **Dijkstra's shortest path algorithm**.

Dijkstra's algorithm is designed for finding the shortest path in a weighted graph where the edge weights are non-negative. This matches our Level 1 problem because every trail has a positive travel time.

The algorithm works by maintaining the best known travel time to each node.

Initially:

* The start node `A` has a distance of `0`.
* Every other node has an unknown distance, represented by infinity.

The algorithm then repeatedly selects the unvisited node with the smallest known distance and examines its neighbours.

For each neighbour, we calculate:

$$
\text{new distance}
===================

\text{current distance}
+
\text{edge weight}
$$

If this new distance is smaller than the previously recorded distance, we update the distance.

We also record the node that gave us the cheaper route. This allows us to reconstruct the actual route once the destination `B` has been reached.

---

## 2.1 Initialising the Algorithm

We first create two dictionaries.

The first dictionary, `distances`, stores the cheapest known travel time from `A` to every node.

The second dictionary, `previous`, stores the node from which each node was reached.

For example, if we eventually discover that the cheapest way to reach `E` is through `D`, we will store:

```text
previous["E"] = "D"
```

This will later allow us to trace the shortest route backwards from `B` to `A`.

We also use a priority queue so that the node with the smallest current distance is processed first.


In [8]:
import heapq

## 2.2 Initialising Distances

Every node initially has an unknown distance from the start.

We represent an unknown distance using infinity.

The start node is different because we already know that the cost of reaching `A` from `A` is zero.

Therefore:

$$
d(A) = 0
$$

and every other node starts with:

$$
d(v) = \infty
$$

where `v` represents any other node in the graph.


In [ ]:
distances = {}

for node in level1_graph:
    distances[node] = float("inf")

distances[start_node] = 0

distances

{'level': inf,
 'nodes': inf,
 'edges': inf,
 'start': inf,
 'end': inf,
 'adjacency_list': inf,
 'A': 0}

## 2.3 Tracking the Previous Node

Finding the shortest distance is not enough.

The challenge requires us to submit the **sequence of nodes** making up the route.

For this reason, we also maintain a `previous` dictionary.

Initially, we do not know how any node was reached, so every value is `None`.

As Dijkstra's algorithm discovers better routes, these values will be updated.

For example, if the algorithm determines that `D` is the cheapest way to reach `E`, we will store:

```text
E → D
```

in the `previous` dictionary.

Once we reach `B`, we can follow these previous-node relationships backwards to reconstruct the complete route.


In [10]:
previous = {}

for node in level1_graph:
    previous[node] = None

previous

{'level': None,
 'nodes': None,
 'edges': None,
 'start': None,
 'end': None,
 'adjacency_list': None}

## 2.4 Creating the Priority Queue

Dijkstra's algorithm must always process the node with the smallest known distance.

Python's `heapq` module provides a priority queue that is useful for this.

We store entries in the form:

```text
(distance, node)
```

At the beginning, the only node we know how to reach is the start node `A`, with a distance of `0`.

Therefore, our initial priority queue contains:

```text
(0, A)
```


In [11]:
priority_queue = [(0, start_node)]

priority_queue

[(0, 'A')]

## 2.5 Running Dijkstra's Algorithm

We can now combine the previous steps into the main Dijkstra loop.

While there are still nodes in the priority queue, we remove the node with the smallest known distance.

For each neighbour of that node, we calculate the cost of travelling to the neighbour.

Because this is Level 1, the edge's `weight` is the travel cost.

The relaxation step is:

$$
\text{new distance}
===================

\text{current distance}
+
\text{edge weight}
$$

If the new distance is smaller than the previously recorded distance, we update both:

1. The distance to the neighbour.
2. The previous node used to reach the neighbour.

We can stop once `B` is removed from the priority queue because Dijkstra guarantees that the first time the destination is selected, its shortest distance has been determined.


In [18]:
import heapq

distances = {
    node: float("inf")
    for node in level1_graph
}

previous = {
    node: None
    for node in level1_graph
}

distances[start_node] = 0

priority_queue = [(0, start_node)]

while priority_queue:

    current_distance, current_node = heapq.heappop(priority_queue)

    # Ignore an outdated entry in the priority queue.
    if current_distance > distances[current_node]:
        continue

    # Stop once the destination has been reached.
    if current_node == end_node:
        break

    # Examine each neighbour.
    for neighbour_info in level1_graph[current_node]:

        neighbour = neighbour_info["node"]
        weight = neighbour_info["weight"]

        new_distance = current_distance + weight

        # Check whether this is a better route.
        if new_distance < distances[neighbour]:

            distances[neighbour] = new_distance
            previous[neighbour] = current_node

            heapq.heappush(
                priority_queue,
                (new_distance, neighbour)
            )

In [19]:
print("Distances:")
print(distances)

print()

print("Previous nodes:")
print(previous)

Distances:
{'A': 0, 'B': 9, 'C': 3, 'D': 2, 'E': 5, 'F': 7}

Previous nodes:
{'A': None, 'B': 'E', 'C': 'D', 'D': 'A', 'E': 'D', 'F': 'E'}


In [20]:
route = []

current_node = end_node

while current_node is not None:

    route.append(current_node)

    if current_node == start_node:
        break

    current_node = previous[current_node]

route.reverse()

print("Route:", route)
print("Cost:", distances[end_node])

Route: ['A', 'D', 'E', 'B']
Cost: 9


# 3. Output File

The hackathon requires the final answer to be uploaded as a **TXT file**.

Although the file extension is `.txt`, the contents of the file must still follow the required JSON format.

For Level 1, the required format is:

```json
{
    "route": ["A", "D", "E", "B"]
}
```

We therefore take the route produced by Dijkstra's algorithm and write it into a text file called `level1_submission.txt`.

The total route cost is **not** included in the submission because the scoring system calculates the cost from the submitted route.


In [ ]:
import json

output = {
    "route": route
}

with open("level1_submission.txt", "w") as file:
    json.dump(output, file, indent=4)

## 3.1 Checking the Output File

We can read the TXT file back into Python to make sure that it contains valid JSON and that the route is correct.

This is an important final check before uploading the file to the hackathon website.


In [22]:
with open("level1_submission.json", "r") as file:
    submission = json.load(file)

print(submission)

{'route': ['A', 'D', 'E', 'B']}


## Final validation

In [23]:
assert submission["route"][0] == start_node
assert submission["route"][-1] == end_node

print("Output file is valid.")
print("Route:", submission["route"])
print("Cost:", distances[end_node])

Output file is valid.
Route: ['A', 'D', 'E', 'B']
Cost: 9


## 3.2 Final Submission

The resulting file is:

```text
level1_submission.json
```

Its contents should be:

```json
{
    "route": [
        "A",
        "D",
        "E",
        "B"
    ]
}
```

Only the route is submitted. The calculated cost of `9` is useful for checking our algorithm, but it is not included in the JSON file because the hackathon scoring system calculates the route cost automatically.

The complete Level 1 workflow is therefore:

**Input**

Load `1.txt` and extract the adjacency list.

↓

**Processing**

Use Dijkstra's algorithm to find the minimum-cost route from `A` to `B`.

↓

**Output**

Save the resulting route as `level1_submission.json`.
